# Nguyễn Thành Trí - MSSV: 6451071079
## Phần 2 - Bài 5a (Trang 45) - Câu 10: Tạo luật kết hợp với min_conf = 50% và in ra các luật

### Đề bài:
Từ các tập phổ biến tìm được ở Câu 9, sinh các luật kết hợp với ngưỡng độ tin cậy tối thiểu `min_conf = 50%` (`min_threshold = 0.5`) và in danh sách các luật theo định dạng trực quan.

In [1]:
# Nguyễn Thành Trí - 6451071079
# Phần 2 - Bài 5a (Trang 45) - Câu 10: Sinh luật kết hợp với min_conf = 50%

import os
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules

print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

def load_clean_data():
    pkl_candidates = ["datasets/online_retail_clean.pkl", "../datasets/online_retail_clean.pkl", "../../datasets/online_retail_clean.pkl"]
    for p in pkl_candidates:
        if os.path.exists(p):
            return pd.read_pickle(p)
    file_path = "datasets/Online Retail.xlsx" if os.path.exists("datasets/Online Retail.xlsx") else "../datasets/Online Retail.xlsx"
    df = pd.read_excel(file_path)
    df['Description'] = df['Description'].str.strip()
    df = df.dropna(subset=['InvoiceNo'])
    df['InvoiceNo'] = df['InvoiceNo'].astype(str)
    return df[~df['InvoiceNo'].str.contains('C', na=False)]

df = load_clean_data()
df_germany = df[df['Country'] == 'Germany']

basket = (df_germany.groupby(['InvoiceNo', 'Description'])['Quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('InvoiceNo'))

basket_sets = (basket >= 1)
if 'POSTAGE' in basket_sets.columns:
    basket_sets = basket_sets.drop(columns=['POSTAGE'])

frequent_itemsets = apriori(basket_sets, min_support=0.05, use_colnames=True)

# Tạo luật kết hợp với min_conf = 50% (0.5)
min_conf = 0.5
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_conf)

# Sắp xếp các luật theo độ tin cậy (confidence) và độ nâng (lift) giảm dần
rules = rules.sort_values(by=['confidence', 'lift'], ascending=[False, False]).reset_index(drop=True)

print(f"Tổng số luật kết hợp tìm được với min_conf = {min_conf*100:.0f}%: {len(rules)} luật\n")

# Chuyển đổi frozenset sang chuỗi hiển thị
rules_display = rules.copy()
rules_display['antecedents_str'] = rules_display['antecedents'].apply(lambda x: ', '.join(list(x)))
rules_display['consequents_str'] = rules_display['consequents'].apply(lambda x: ', '.join(list(x)))

print("DANH SÁCH CHI TIẾT CÁC LUẬT KẾT HỢP (THỊ TRƯỜNG ĐỨC):")
for i, r in rules_display.iterrows():
    print(f"Luật {i+1}: {r['antecedents_str']}  ==>  {r['consequents_str']}")
    print(f"         [Support: {r['support']:.4f} ({r['support']*100:.2f}%) | Confidence: {r['confidence']:.4f} ({r['confidence']*100:.2f}%) | Lift: {r['lift']:.4f}]")

print("\nBẢNG TỔNG HỢP CÁC CHỈ SỐ CỦA 8 LUẬT KẾT HỢP:")
display(rules_display[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift', 'leverage', 'conviction']])

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079
Tổng số luật kết hợp tìm được với min_conf = 50%: 8 luật

DANH SÁCH CHI TIẾT CÁC LUẬT KẾT HỢP (THỊ TRƯỜNG ĐỨC):
Luật 1: RED RETROSPOT CHARLOTTE BAG  ==>  WOODLAND CHARLOTTE BAG
         [Support: 0.0591 (5.91%) | Confidence: 0.8438 (84.38%) | Lift: 6.6482]
Luật 2: ROUND SNACK BOXES SET OF 4 FRUITS  ==>  ROUND SNACK BOXES SET OF4 WOODLAND
         [Support: 0.1313 (13.13%) | Confidence: 0.8333 (83.33%) | Lift: 3.4003]
Luật 3: SPACEBOY LUNCH BOX  ==>  ROUND SNACK BOXES SET OF4 WOODLAND
         [Support: 0.0700 (7.00%) | Confidence: 0.6809 (68.09%) | Lift: 2.7781]
Luật 4: PLASTERS IN TIN CIRCUS PARADE  ==>  PLASTERS IN TIN WOODLAND ANIMALS
         [Support: 0.0678 (6.78%) | Confidence: 0.5849 (58.49%) | Lift: 4.2429]
Luật 5: PLASTERS IN TIN SPACEBOY  ==>  PLASTERS IN TIN WOODLAND ANIMALS
         [Support: 0.0613 (6.13%) | Confidence: 0.5714 (57.14%) | Lift: 4.1451]
Luật 6: PLASTERS IN TIN WOODLAND ANIMALS  ==>  ROUND SNACK BOXES SET OF4 WOODL

,antecedents_str,consequents_str,support,confidence,lift,leverage,conviction
0,RED RETROSPOT CHARLOTTE BAG,WOODLAND CHARLOTTE BAG,0.059081,0.843750,6.648168,0.050194,5.587746
1,ROUND SNACK BOXES SET OF 4 FRUITS,ROUND SNACK BOXES SET OF4 WOODLAND,0.131291,0.833333,3.400298,0.092679,4.529540
2,SPACEBOY LUNCH BOX,ROUND SNACK BOXES SET OF4 WOODLAND,0.070022,0.680851,2.778116,0.044817,2.365427
3,PLASTERS IN TIN CIRCUS PARADE,PLASTERS IN TIN WOODLAND ANIMALS,0.067834,0.584906,4.242887,0.051846,2.076984
4,PLASTERS IN TIN SPACEBOY,PLASTERS IN TIN WOODLAND ANIMALS,0.061269,0.571429,4.145125,0.046488,2.011670
5,PLASTERS IN TIN WOODLAND ANIMALS,ROUND SNACK BOXES SET OF4 WOODLAND,0.074398,0.539683,2.202098,0.040613,1.640006
6,ROUND SNACK BOXES SET OF4 WOODLAND,ROUND SNACK BOXES SET OF 4 FRUITS,0.131291,0.535714,3.400298,0.092679,1.814509
7,WOODLAND CHARLOTTE BAG,ROUND SNACK BOXES SET OF4 WOODLAND,0.063457,0.500000,2.040179,0.032354,1.509847


### Nhận xét & Đánh giá (Câu 10):
1. **Số lượng và chất lượng các luật**: Thuật toán sinh ra đúng **8 luật kết hợp mạnh**. Đáng chú ý, **100% các luật đều có Lift > 2.0** (dao động từ 2.04 đến 6.65), chứng minh các mặt hàng có mối liên hệ mua kèm thực sự rất chặt chẽ chứ không phải do xác suất ngẫu nhiên.
2. **Phân tích các luật nổi bật nhất**:
   - **Luật có Lift cao nhất**: `{RED RETROSPOT CHARLOTTE BAG} ==> {WOODLAND CHARLOTTE BAG}`:
     - `Confidence = 84.38%`, `Lift = 6.65`, `Support = 5.91%`.
     - Ý nghĩa: Trong số các khách hàng Đức đã mua túi Charlotte chấm bi đỏ, có tới **84.38%** khách hàng sẽ mua kèm theo túi Charlotte hình thú rừng Woodland. Độ nâng `Lift = 6.65` cho thấy việc mua túi chấm bi đỏ làm tăng khả năng mua túi Woodland lên gấp **6.65 lần**!
   - **Cặp luật hai chiều mạnh nhất về Support**: Bộ hộp đựng đồ ăn nhẹ (`ROUND SNACK BOXES`):
     - `{ROUND SNACK BOXES SET OF 4 FRUITS} ==> {ROUND SNACK BOXES SET OF4 WOODLAND}`: `Confidence = 83.33%`, `Lift = 3.40`, `Support = 13.13%`.
     - `{ROUND SNACK BOXES SET OF4 WOODLAND} ==> {ROUND SNACK BOXES SET OF 4 FRUITS}`: `Confidence = 53.57`, `Lift = 3.40`, `Support = 13.13%`.
     - Xuất hiện trong hơn 13% tổng đơn hàng của Đức, khẳng định đây là combo sản phẩm gia đình thiết yếu được mua cùng nhau thường xuyên nhất.
   - **Dòng sản phẩm hộp băng cá nhân (`PLASTERS IN TIN`)**:
     - `{PLASTERS IN TIN CIRCUS PARADE} ==> {PLASTERS IN TIN WOODLAND ANIMALS}` (`Conf = 58.49%`, `Lift = 4.24`).
     - `{PLASTERS IN TIN SPACEBOY} ==> {PLASTERS IN TIN WOODLAND ANIMALS}` (`Conf = 57.14%`, `Lift = 4.15`).
     - Khách hàng có xu hướng sưu tập trọn bộ các mẫu hình dán băng cá nhân cho trẻ nhỏ.